In [21]:
from typing import Any
import os
import pandas as pd
import psycopg2
from dotenv import load_dotenv
from psycopg2.extras import RealDictCursor
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever
load_dotenv()

True

In [20]:
api_key = os.getenv("OPENAI_API_KEY")

In [3]:
def get_db_connection():
    try:
        conn = psycopg2.connect(
            host=os.getenv("DB_HOST"),
            port=os.getenv("DB_PORT"),
            database=os.getenv("DB_NAME"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
        )
        print("[Debug] DB 연결 성공")
        return conn

    except Exception as e:
        print(f"[Error] DB 연결 실패: {e}")
        raise

In [4]:
class EventRepository:
    def load_events_dataframe(self) -> pd.DataFrame:
        query = """
            SELECT payload::jsonb -> 'source_row' AS source_row
            FROM public.events
            WHERE jsonb_typeof(payload::jsonb -> 'source_row') = 'object'
        """

        conn = get_db_connection()

        try:
            with conn.cursor(cursor_factory=RealDictCursor) as cur:
                cur.execute(query)
                rows: list[dict[str, Any]] = cur.fetchall()

            source_rows = [row["source_row"] for row in rows]
            event_df = pd.DataFrame(source_rows)

            print(f"[Debug] 조회된 행사: {len(event_df)}개")
            return event_df

        finally:
            conn.close()

In [34]:
df = EventRepository().load_events_dataframe()

[Debug] DB 연결 성공
[Debug] 조회된 행사: 222개


In [35]:
import pandas as pd

# 1. 수정 전 데이터 확인 (장소에 'Osaka'가 포함된 데이터 출력)
print("--- 수정 전 ---")
display(df[df['장소'].str.contains('Osaka', na=False)][['Index', '장소', '제목']])

# 2. '장소' 컬럼이 "Osaka, Japan"인 값을 "Osaka"로 변경
df.loc[df['장소'] == 'Osaka, Japan', '장소'] = 'Osaka'

# (참고) 만약 'Osaka, Japan' 외에도 콤마(,)가 포함된 문자열에서 지역명만 남기고 싶다면 아래처럼 정규식을 쓸 수도 있습니다.
# df['장소'] = df['장소'].astype(str).str.replace(r'Osaka, Japan', 'Osaka', regex=False)

# 3. 수정 후 데이터 확인
print("\n--- 수정 후 ---")
display(df[df['장소'].str.contains('Osaka', na=False)][['Index', '장소', '제목']])

--- 수정 전 ---


,Index,장소,제목
18,19,"Osaka, Japan",2026 5th International Conference on Power Sys...
60,60,"Osaka, Japan",2026 13th International Conference on Power an...
121,122,"Osaka, Japan",2026 2nd International Conference on Power Eng...



--- 수정 후 ---


,Index,장소,제목
18,19,Osaka,2026 5th International Conference on Power Sys...
60,60,Osaka,2026 13th International Conference on Power an...
121,122,Osaka,2026 2nd International Conference on Power Eng...


In [7]:
def create_documents_from_dataframe(df: pd.DataFrame) -> list[Document]:
    EXCLUDE_FROM_CONTENT = {
        "Index", "등록 링크", "상세 정보 링크", "시작 일시", "종료 일시", 
        "유료 여부", "첨부파일 유무", "접근성 상태", "출처"
    }
    docs = []
    all_columns = set(df.columns)
    content_columns = list(all_columns - EXCLUDE_FROM_CONTENT)

    for _, row in df.iterrows():
        content_parts = []
        for col in content_columns:
            val = row.get(col)
            if pd.notna(val) and str(val).strip() != "":
                content_parts.append(f"{col}: {val}")

        page_content = "\n".join(content_parts)
        metadata = {
            "index": row.get("Index"),
            "start_date": str(row.get("시작 일시")) if pd.notna(row.get("시작 일시")) else None,
            "end_date": str(row.get("종료 일시")) if pd.notna(row.get("종료 일시")) else None,
            "organizer": row.get("주최") if pd.notna(row.get("주최")) else None,
            "location": row.get("장소") if pd.notna(row.get("장소")) else None,
        }
        docs.append(Document(page_content=page_content, metadata=metadata))

    return docs

In [36]:
documents = create_documents_from_dataframe(df)

In [37]:
# dense retriever
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=api_key)
vectorstore = FAISS.from_documents(documents, embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k":5})

In [38]:
#sparse retriever
sparse_retriever = BM25Retriever.from_documents(documents)
sparse_retriever.k = 5

In [39]:
#hybrid
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.5, 0.5]
)

In [43]:
query = "일본에서 2월에 개최되는 인공지능 관련 행사"
query2 = "일본에서 2월에 개최되는 행사"
query3 = "일본"

In [44]:
dense_retriever.invoke(query3)

[Document(id='9e2bf661-d472-47bc-9de2-7da781641f96', metadata={'index': 126, 'start_date': '2026-10-28 00:00:00 GMT (날짜 전용)', 'end_date': '2026-10-28 23:59:59 GMT (날짜 전용)', 'organizer': 'OECD Nuclear Energy Agency', 'location': '미기재'}, page_content='주최: OECD Nuclear Energy Agency\n장소: 미기재\n주제 요약: OECD NEA 공개 generated.Event 검색 결과\n제목: NEA Forum on Stakeholder Confidence (FSC) Japanese National Workshop 2026\n주요 키워드: OECD NEA, 원자력\n행사 성격: 행사·교육'),
 Document(id='bea0e214-96c6-4aa5-83b2-c5adc4e01e5d', metadata={'index': 86, 'start_date': '2026-10-12 00:00:00 GMT (날짜 전용)', 'end_date': '2026-10-16 23:59:59 GMT (날짜 전용)', 'organizer': '한국원자력산업협회', 'location': 'Tokyo, Japan |'}, page_content='주최: 한국원자력산업협회\n장소: Tokyo, Japan |\n주제 요약: KAIF 원자력계 일정의 공개 행사\n제목: TopFuel 2026\n주요 키워드: 원자력산업, KAIF\n행사 성격: 세미나·행사'),
 Document(id='6a4f2a50-38bd-41f4-b8dc-6429f7231039', metadata={'index': 194, 'start_date': '2027-02-26 00:00:00 GMT (날짜 전용)', 'end_date': '2027-02-26 23:59:59 GMT (날짜 전용)', 'organizer': '

In [41]:
dense_retriever.invoke(query)

[Document(id='b3c9c51e-3234-4536-ae23-58e225b00edd', metadata={'index': 186, 'start_date': '2026-12-18 00:00:00 GMT', 'end_date': '2026-12-20 23:59:59 GMT', 'organizer': 'IEEE PES (재정 또는 기술 후원)', 'location': 'Beijing, China'}, page_content='주최: IEEE PES (재정 또는 기술 후원)\n장소: Beijing, China\n주제 요약: 인공지능의 전력계통 해석·제어 적용과 실습 사례를 다룹니다.\n제목: 2026 2nd International Conference on Power Systems, Smart Grid, and Artificial Intelligence (PSGAI)\n주요 키워드: 인공지능, 전력계통, 계통 해석·제어\n행사 성격: 컨퍼런스'),
 Document(id='b0e43a23-479c-4590-928d-931bdc6f4c4e', metadata={'index': 145, 'start_date': '2026-11-09 07:00:00 GMT', 'end_date': '2026-11-12 17:00:00 GMT', 'organizer': 'IAEA', 'location': 'College Station, TX, USA'}, page_content='주최: IAEA\n장소: College Station, TX, USA\n주제 요약: IAEA Indico 공개 미래 행사 목록\n제목: Second Workshop on AI for Accelerating Fusion Energy and Plasma Science (AI for Fusion)\n주요 키워드: IAEA, 원자력\n행사 성격: 워크숍'),
 Document(id='b82beda1-aefa-4e36-b07d-b63073dfb50b', metadata={'index': 13, 'start_date'

In [30]:
sparse_retriever.invoke(query)

[Document(metadata={'index': 214, 'start_date': '2027-09-05 00:00:00 GMT', 'end_date': '2027-09-09 23:59:59 GMT', 'organizer': 'IEEE PES (재정 또는 기술 후원)', 'location': 'Bruges, Belgium'}, page_content='주최: IEEE PES (재정 또는 기술 후원)\n장소: Bruges, Belgium\n주제 요약: 2027 Bruges PowerTech 관련 IEEE PES 공개 행사입니다.\n제목: 2027 Bruges PowerTech\n주요 키워드: 전력·에너지, 국제 행사\n행사 성격: 공개 행사'),
 Document(metadata={'index': 219, 'start_date': '2027-11-07 00:00:00 GMT', 'end_date': '2027-11-11 23:59:59 GMT', 'organizer': 'IEEE PES (재정 또는 기술 후원)', 'location': 'Dakar, Senegal'}, page_content='주최: IEEE PES (재정 또는 기술 후원)\n장소: Dakar, Senegal\n주제 요약: 2027 IEEE PES/IAS PowerAfrica 관련 IEEE PES 공개 행사입니다.\n제목: 2027 IEEE PES/IAS PowerAfrica\n주요 키워드: 전력·에너지, 국제 행사\n행사 성격: 공개 행사'),
 Document(metadata={'index': 59, 'start_date': '2026-09-21 00:00:00 GMT', 'end_date': '2026-09-25 23:59:59 GMT', 'organizer': 'IEEE PES (재정 또는 기술 후원)', 'location': 'Nairobi, Kenya'}, page_content='주최: IEEE PES (재정 또는 기술 후원)\n장소: Nairobi, Kenya\n주제 요약: 20

In [31]:
hybrid_retriever.invoke(query)

[Document(id='6e7531b4-4977-4656-bee0-c75def0e056e', metadata={'index': 186, 'start_date': '2026-12-18 00:00:00 GMT', 'end_date': '2026-12-20 23:59:59 GMT', 'organizer': 'IEEE PES (재정 또는 기술 후원)', 'location': 'Beijing, China'}, page_content='주최: IEEE PES (재정 또는 기술 후원)\n장소: Beijing, China\n주제 요약: 인공지능의 전력계통 해석·제어 적용과 실습 사례를 다룹니다.\n제목: 2026 2nd International Conference on Power Systems, Smart Grid, and Artificial Intelligence (PSGAI)\n주요 키워드: 인공지능, 전력계통, 계통 해석·제어\n행사 성격: 컨퍼런스'),
 Document(metadata={'index': 214, 'start_date': '2027-09-05 00:00:00 GMT', 'end_date': '2027-09-09 23:59:59 GMT', 'organizer': 'IEEE PES (재정 또는 기술 후원)', 'location': 'Bruges, Belgium'}, page_content='주최: IEEE PES (재정 또는 기술 후원)\n장소: Bruges, Belgium\n주제 요약: 2027 Bruges PowerTech 관련 IEEE PES 공개 행사입니다.\n제목: 2027 Bruges PowerTech\n주요 키워드: 전력·에너지, 국제 행사\n행사 성격: 공개 행사'),
 Document(id='b8ad8a86-4962-426a-b770-2cb3c5bdb8e8', metadata={'index': 145, 'start_date': '2026-11-09 07:00:00 GMT', 'end_date': '2026-11-12 17:00:00 GM

In [42]:
dense_retriever.invoke(query2)
#sparse_retriever.invoke(query2)
#hybrid_retriever.invoke(query2)

[Document(id='bea0e214-96c6-4aa5-83b2-c5adc4e01e5d', metadata={'index': 86, 'start_date': '2026-10-12 00:00:00 GMT (날짜 전용)', 'end_date': '2026-10-16 23:59:59 GMT (날짜 전용)', 'organizer': '한국원자력산업협회', 'location': 'Tokyo, Japan |'}, page_content='주최: 한국원자력산업협회\n장소: Tokyo, Japan |\n주제 요약: KAIF 원자력계 일정의 공개 행사\n제목: TopFuel 2026\n주요 키워드: 원자력산업, KAIF\n행사 성격: 세미나·행사'),
 Document(id='75cac86d-f3e0-407c-aa4f-c9c27999b38d', metadata={'index': 41, 'start_date': '2026-09-09 00:00:00 GMT (날짜 전용)', 'end_date': '2026-09-11 23:59:59 GMT (날짜 전용)', 'organizer': '한국원자력산업협회', 'location': 'Tokyo, Japan |'}, page_content='주최: 한국원자력산업협회\n장소: Tokyo, Japan |\n주제 요약: KAIF 원자력계 일정의 공개 행사\n제목: Smart Energy Week 2026\n주요 키워드: 원자력산업, KAIF\n행사 성격: 세미나·행사'),
 Document(id='f5792361-70aa-43f5-8ab0-13ee83f30280', metadata={'index': 27, 'start_date': '2026-08-24 00:00:00 GMT (날짜 전용)', 'end_date': '2026-08-27 23:59:59 GMT (날짜 전용)', 'organizer': '한국원자력산업협회', 'location': 'Dallas, TX |'}, page_content='주최: 한국원자력산업협회\n장소: Dallas